# Healthcare Administrative Semantic Routing

This notebook routes synthetic member and provider requests across seven administrative workflows: claims, eligibility, prior authorization, benefits, pharmacy, infusion therapy, and appeals.

All examples are synthetic and contain no protected health information (PHI). This example does not provide clinical advice, make coverage decisions, authorize care, adjudicate claims, or establish regulatory compliance.

## Setup

Run the notebook from the repository root after installing the local extra with Python 3.12 or earlier:

```bash
uv sync --extra local
```

`LocalEncoder` performs inference on the local machine. The first run may download model artifacts.

In [1]:
import csv
import random
from collections import Counter
from pathlib import Path

import numpy as np

from semantic_router import Route, SemanticRouter
from semantic_router.encoders import LocalEncoder

## Define the seven routes

The route names identify operational destinations. The utterances emphasize the requested administrative action so that shared terms such as *denied*, *coverage*, *medication*, and *infusion* do not determine a route by themselves.

In [2]:
claims = Route(
    name="claims",
    utterances=[
        "Check whether my medical claim was received",
        "What is the status of the claim we submitted",
        "Explain how this claim was paid",
        "We need to correct a billed claim",
        "Has the plan reprocessed the claim",
        "Show the remittance information for this claim",
        "The service was completed and I need the claim payment status",
        "Check the billed claim after care was provided",
    ],
)

eligibility = Route(
    name="eligibility",
    utterances=[
        "Confirm that my enrollment is active",
        "Verify member eligibility for the service date",
        "When does this coverage become effective",
        "What is the coverage termination date",
        "Was my dependent added to the plan",
        "The eligibility response shows inactive status",
        "Verify eligibility even though a claim has been submitted",
        "Was the member enrolled on the date care was received",
    ],
)

prior_authorization = Route(
    name="prior_authorization",
    utterances=[
        "Does this service require approval before it is performed",
        "Check the status of the authorization request",
        "Submit clinical records for the pending authorization",
        "Start a prior authorization for this procedure",
        "The medication needs prior approval",
        "Extend the approved dates on the authorization",
        "Issue an authorization number before the infusion is scheduled",
        "Has the upcoming procedure been approved in advance",
    ],
)

benefits = Route(
    name="benefits",
    utterances=[
        "What does my plan cover",
        "Explain the deductible for this service",
        "How many visits does the benefit allow",
        "What is the copay for specialist care",
        "Does the member have out-of-network benefits",
        "Is this service excluded from the plan",
        "The member is active and wants to know whether this treatment is covered",
        "Does the plan pay for a second medical opinion",
    ],
)

pharmacy = Route(
    name="pharmacy",
    utterances=[
        "Is this prescription on the formulary",
        "Which pharmacy can dispense the medication",
        "Why is the refill too soon",
        "Request a vacation refill override",
        "What quantity limit applies to this drug",
        "Transfer the prescription to a specialty pharmacy",
        "Which specialty pharmacy must dispense this prescription",
        "Explain the refill limit for this medicine",
    ],
)

infusion_therapy = Route(
    name="infusion_therapy",
    utterances=[
        "Schedule the next infusion appointment",
        "Which site can administer the infusion",
        "Reschedule my infusion visit",
        "Confirm delivery of the drug to the infusion center",
        "Move the infusion to an outpatient site",
        "Coordinate the updated infusion dose with the clinic",
        "Authorization is complete so arrange the infusion site and appointment",
        "Approval is on file and the infusion delivery must be coordinated",
    ],
)

appeals = Route(
    name="appeals",
    utterances=[
        "I want to appeal this denial",
        "Submit a request for reconsideration",
        "Challenge the adverse coverage decision",
        "What is the deadline to file an appeal",
        "Check the status of the expedited appeal",
        "Request an independent review of the decision",
        "Reconsider the decision not to cover the medication",
        "Challenge the decision after an authorization was denied",
    ],
)

routes = [
    claims,
    eligibility,
    prior_authorization,
    benefits,
    pharmacy,
    infusion_therapy,
    appeals,
]

## Create a fully local router

`BAAI/bge-small-en-v1.5` is the default model used by `LocalEncoder`. After the model is available, embedding inference does not require an external API.

In [3]:
encoder = LocalEncoder(name="BAAI/bge-small-en-v1.5")
router = SemanticRouter(
    encoder=encoder,
    routes=routes,
    auto_sync="local",
)

print(f"Encoder: {encoder.name}")
print(f"Device: {encoder.device}")

Encoder: BAAI/bge-small-en-v1.5
Device: cpu


## Load the synthetic evaluation dataset

Empty `expected_route` values are converted to `None`, which means the router should abstain. Route seed utterances above are not duplicated in this dataset.

In [4]:
candidate_paths = [
    Path("healthcare-routing-evaluation.csv"),
    Path("docs/examples/healthcare-routing/healthcare-routing-evaluation.csv"),
]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find healthcare-routing-evaluation.csv")

with data_path.open(encoding="utf-8", newline="") as dataset_file:
    dataset = list(csv.DictReader(dataset_file))

for row in dataset:
    row["expected_route"] = row["expected_route"] or None

def rows_for(split):
    return [row for row in dataset if row["split"] == split]

def xy(rows):
    return (
        [row["utterance"] for row in rows],
        [row["expected_route"] for row in rows],
    )

for split in ("train", "validation", "test"):
    split_rows = rows_for(split)
    print(split, len(split_rows), Counter(row["example_type"] for row in split_rows))

train 55 Counter({'positive': 49, 'negative': 6})
validation 24 Counter({'positive': 14, 'ambiguous': 7, 'negative': 3})
test 24 Counter({'positive': 14, 'ambiguous': 7, 'negative': 3})


## Evaluate default thresholds

The built-in `evaluate` method reports accuracy. We evaluate the validation split before fitting so that the effect of threshold optimization is visible.

In [5]:
validation_x, validation_y = xy(rows_for("validation"))
baseline_validation_accuracy = router.evaluate(X=validation_x, y=validation_y)

print(f"Baseline validation accuracy: {baseline_validation_accuracy:.1%}")
print("Default thresholds:", router.get_thresholds())

Baseline validation accuracy: 54.2%
Default thresholds: {'claims': 0.0, 'eligibility': 0.0, 'prior_authorization': 0.0, 'benefits': 0.0, 'pharmacy': 0.0, 'infusion_therapy': 0.0, 'appeals': 0.0}


## Optimize route thresholds

Only the training split is passed to `fit`. Validation and test rows remain held out. The training data includes unrelated negative examples so the optimization can learn to abstain.

In [6]:
train_x, train_y = xy(rows_for("train"))
random.seed(42)
np.random.seed(42)
router.fit(X=train_x, y=train_y, max_iter=500)

optimized_validation_accuracy = router.evaluate(X=validation_x, y=validation_y)
print("Optimized thresholds:", router.get_thresholds())
print(f"Optimized validation accuracy: {optimized_validation_accuracy:.1%}")

Optimized thresholds: {'claims': 0.6577900060949944, 'eligibility': 0.6684215896337109, 'prior_authorization': 0.6464646464646465, 'benefits': 0.494949494949495, 'pharmacy': 0.5309458218549128, 'infusion_therapy': 0.7546168758289971, 'appeals': 0.6442199775533108}
Optimized validation accuracy: 54.2%


## Evaluate the held-out test split

Accuracy alone can hide unsafe behavior. The helper below also reports per-route precision, recall, and F1; abstention performance; and coverage. In this example, `None` is the abstention label.

In [7]:
def predict(rows):
    return [router(row["utterance"]).name for row in rows]

def classification_report(expected, predicted):
    labels = [route.name for route in routes] + [None]
    report = {}
    for label in labels:
        true_positive = sum(e == label and p == label for e, p in zip(expected, predicted))
        false_positive = sum(e != label and p == label for e, p in zip(expected, predicted))
        false_negative = sum(e == label and p != label for e, p in zip(expected, predicted))
        support = sum(e == label for e in expected)
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        report[label or "abstain"] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    return report

test_rows = rows_for("test")
test_expected = [row["expected_route"] for row in test_rows]
test_predicted = predict(test_rows)
test_accuracy = sum(e == p for e, p in zip(test_expected, test_predicted)) / len(test_rows)
coverage = sum(prediction is not None for prediction in test_predicted) / len(test_predicted)
report = classification_report(test_expected, test_predicted)

print(f"Test accuracy: {test_accuracy:.1%}")
print(f"Coverage: {coverage:.1%}")
for label, metrics in report.items():
    print(
        f"{label:20} precision={metrics['precision']:.2f} "
        f"recall={metrics['recall']:.2f} f1={metrics['f1']:.2f} "
        f"support={metrics['support']}"
    )

Test accuracy: 83.3%
Coverage: 95.8%
claims               precision=0.67 recall=0.67 f1=0.67 support=3
eligibility          precision=0.67 recall=0.67 f1=0.67 support=3
prior_authorization  precision=1.00 recall=1.00 f1=1.00 support=3
benefits             precision=0.60 recall=1.00 f1=0.75 support=3
pharmacy             precision=1.00 recall=1.00 f1=1.00 support=3
infusion_therapy     precision=1.00 recall=1.00 f1=1.00 support=3
appeals              precision=1.00 recall=1.00 f1=1.00 support=3
abstain              precision=1.00 recall=0.33 f1=0.50 support=3


In [8]:
confusion = Counter(zip(test_expected, test_predicted))
for (expected, predicted), count in sorted(
    confusion.items(),
    key=lambda item: (str(item[0][0]), str(item[0][1])),
):
    print(f"expected={expected or 'abstain':20} predicted={predicted or 'abstain':20} count={count}")

expected=abstain              predicted=abstain              count=1
expected=abstain              predicted=benefits             count=1
expected=abstain              predicted=claims               count=1
expected=appeals              predicted=appeals              count=3
expected=benefits             predicted=benefits             count=3
expected=claims               predicted=claims               count=2
expected=claims               predicted=eligibility          count=1
expected=eligibility          predicted=benefits             count=1
expected=eligibility          predicted=eligibility          count=2
expected=infusion_therapy     predicted=infusion_therapy     count=3
expected=pharmacy             predicted=pharmacy             count=3
expected=prior_authorization  predicted=prior_authorization  count=3


## Inspect ambiguous requests

Some ambiguous examples have a label because the requested action supplies a precedence signal. Others intentionally expect abstention. `limit=3` exposes up to three routes that pass their thresholds; it does not turn similarity scores into probabilities.

In [9]:
ambiguous_rows = [row for row in dataset if row["example_type"] == "ambiguous"]
for row in ambiguous_rows:
    choices = router(row["utterance"], limit=3)
    if not isinstance(choices, list):
        choices = [choices]
    candidates = [
        (choice.name, choice.similarity_score)
        for choice in choices
        if choice.name is not None
    ]
    print(
        f"{row['id']} expected={row['expected_route'] or 'abstain'} "
        f"candidates={candidates}\n  {row['utterance']}"
    )

HC076 expected=abstain candidates=[('appeals', 0.7226199677688235), ('prior_authorization', 0.7176114909185595)]
  My infusion medication was denied.


HC077 expected=abstain candidates=[('benefits', 0.7615165814259329), ('appeals', 0.7572530970717702), ('eligibility', 0.7382934721860266)]
  We need help with a member's coverage.


HC078 expected=abstain candidates=[('pharmacy', 0.7659763044218655), ('prior_authorization', 0.7605185348028739)]
  The pharmacy says there is a problem with my prescription.


HC079 expected=abstain candidates=[('benefits', 0.7713699893774688), ('claims', 0.6998867263932905), ('appeals', 0.685923395032509)]
  The plan did not approve the service.


HC080 expected=abstain candidates=[('benefits', 0.7833040874169865), ('claims', 0.7687580032442362), ('eligibility', 0.7543372563934182)]
  I have a question about my bill and coverage.


...

HC085 expected=prior_authorization candidates=[('prior_authorization', 0.8164224397754111), ('pharmacy', 0.7647210777642464)]
  The pharmacy says prior approval is required; has my doctor submitted it?


HC086 expected=appeals candidates=[('appeals', 0.7926509784782702), ('prior_authorization', 0.7116560677283257), ('claims', 0.7087538253372164)]
  The authorization was denied and we need to request reconsideration.


HC087 expected=infusion_therapy candidates=[('infusion_therapy', 0.8120973336949588), ('prior_authorization', 0.8091482066206034)]
  My infusion authorization is approved; I need to change the appointment.


HC088 expected=pharmacy candidates=[('pharmacy', 0.8405107013476631), ('prior_authorization', 0.7085211532344845)]
  The prescription is covered; which specialty pharmacy should dispense it?


HC089 expected=eligibility candidates=[('benefits', 0.7658164028866893), ('claims', 0.7644434373166877), ('eligibility', 0.7391830898819408)]
  The hospital billed the plan; am I eligible on the service date?


## Map routes to specialized workflows or agents

A routing decision should select a workflow, not directly perform a consequential action. These handlers are placeholders. A production system would add authentication, authorization, audit logging, state checks, and human escalation.

In [10]:
def workflow(name):
    def run(request):
        return {"workflow": name, "request": request, "status": "human_review_required"}
    return run

WORKFLOW_HANDLERS = {
    "claims": workflow("claims_operations"),
    "eligibility": workflow("eligibility_verification"),
    "prior_authorization": workflow("prior_authorization_operations"),
    "benefits": workflow("benefits_explanation"),
    "pharmacy": workflow("pharmacy_benefit_operations"),
    "infusion_therapy": workflow("infusion_coordination"),
    "appeals": workflow("appeals_and_reconsideration"),
}

def route_to_workflow(request):
    choice = router(request)
    if choice.name is None:
        return {
            "workflow": "clarification_or_human_triage",
            "request": request,
            "status": "human_review_required",
        }
    return WORKFLOW_HANDLERS[choice.name](request)

route_to_workflow("My infusion is approved; I need to change the appointment.")

{'workflow': 'infusion_coordination',
 'request': 'My infusion is approved; I need to change the appointment.',
 'status': 'human_review_required'}

## Limitations and extension guidance

- This is a small educational dataset, not a benchmark or production validation.
- The router has no access to conversation history, plan rules, claim state, authorization state, or authenticated identity.
- Similarity scores are not calibrated probabilities, and optimized thresholds can overfit.
- A single-label router cannot fully represent multi-intent requests.
- Local inference does not by itself guarantee privacy, security, or regulatory compliance.
- This example must not be used for emergencies, clinical advice, medical-necessity decisions, care authorization, claim adjudication, or appeal decisions.

To extend the taxonomy, define operational ownership and exclusions first; add independent member and provider seed utterances; create positive, negative, and boundary examples; keep train, validation, and test splits separate; then refit thresholds and review per-route metrics, coverage, and abstention behavior. Version routes, examples, thresholds, and evaluation data together.